In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from ClassGel2d import gel2d
import numpy as np

# ---------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------
L = 100.0
d = 10.0
order = 3
phi0 = 0.7
mu_input = 0.0029
mu_bar = -mu_input

simulation_name = (
    f'test_periodic_L{int(L)}_d{int(d)}'
    f'_phi0={phi0}_nIterations100'
)

# Iteración a visualizar (ejemplo: 0 para la perturbed inicial, 100 para la final)
numIteration = 100

print(f'L={L} [mm], d={d} [mm], order={order}, phi0={phi0}, mu_bar={mu_bar}')

gel = gel2d(L, d, phi0, mu_bar)

mesh_file = 'meshes/meshPeriodic.vol'
mesh = Mesh(mesh_file)

fes = Periodic(VectorH1(mesh, order=order, dirichlet='bot'))

filename = (
    f'{simulation_name}/result_gelsMu2D'
    + gel.filename_suffix
    + '_iter='
    + str(numIteration).zfill(2)
)

print(f'Cargando: {filename}.gfu')

u = GridFunction(fes)
u.Load(filename + '.gfu')

F = Id(2) + Grad(u)
energy_density = Integrate(gel.W(F), mesh)/(L*d)
print('Average energy density = ', energy_density)

# ---------------------------------------------------------
# MEDICIÓN DE AMPLITUD EN LA SUPERFICIE (y = d)
# ---------------------------------------------------------
top_values = []
tol_y = 1e-4  # Tolerancia adecuada para capturar la frontera top en Netgen

for p in mesh.ngmesh.Points():
    x0, y0 = p[0], p[1]
    if abs(y0 - d) < tol_y:
        # Evaluamos el desplazamiento u[1] directamente en el punto (x0, y0)
        u2_val = u[1](mesh(x0, y0))
        y_deformed = y0 + u2_val
        top_values.append(y_deformed)

if top_values:
    y_max = max(top_values)
    y_min = min(top_values)
    surface_amplitude = 0.5 * (y_max - y_min)
    
    print(f'Nodos capturados en superficie superior: {len(top_values)}')
    print(f'Altura máxima y_max = {y_max:.4f} mm')
    print(f'Altura mínima y_min = {y_min:.4f} mm')
    print(f'Amplitud de la superficie = {surface_amplitude:.6f} mm')
    print(f'Amplitud relativa (A/d) = {surface_amplitude/d:.6f}')
else:
    print("ADVERTENCIA: No se encontraron nodos en y = d. Ajusta la tolerancia 'tol_y'.")

# ---------------------------------------------------------
# VISUALIZACIÓN WEBGUI (Escala Real 1:1)
# ---------------------------------------------------------
# Coloreado por desplazamiento vertical u[1] con deformación real aplicada en el visor
Draw(u[1], mesh, deformation=u, name="Desplazamiento_u2")

L=100.0 [mm], d=10.0 [mm], order=3, phi0=0.7, mu_bar=-0.0029
Entropic unit = 137.21 [MPa], G = 0.1372 [MPa], chi = 0.4, p0_bar = 0.0000233, lambda_target = 3.673936794472416
gel.Gfun(lambda_target) = 0.13721349978333267
filename_suffix=_phi0=0.7_absMuBar=0.003
sum(fes.FreeDofs()) = 16674
filename=test_periodic_L100_d10_phi0=0.7_nIterations100/result_gelsMu2D_phi0=0.7_absMuBar=0.003_iter=01.gfu
Average energy density =  nan
Surface amplitude =  nan
Surface amplitude / d =  nan


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…